# Luggage Decision Data — Ingestion

Parses raw Pega decision JSON exports, extracts all luggage weight variant
(L5B15 / L5B20 / L5B25 / L5B30 / L5B40 / L5B50 / CLUG) scoring events for the
**Email / Outbound / Luggage / Sales** partition, and writes a clean Parquet file
to `data/processed/`.

**To add new data:** drop additional JSON export files into `data/decisions/` and re-run.
Deduplication on `modelExecutionID` ensures records are never double-counted.

In [ ]:
from pathlib import Path
import json
from collections import Counter

import pandas as pd
import polars as pl
from IPython.display import HTML, display
from tabulate import tabulate
from tqdm import tqdm

from my_project.parsing import extract_l5b15_rows, LUGGAGE_VARIANTS

print(f"Imports OK — targeting: {sorted(LUGGAGE_VARIANTS)}")

In [ ]:
DECISIONS_DIR = Path("../data/decisions")
PROCESSED_DIR = Path("../data/processed")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_FILE = PROCESSED_DIR / "luggage_email_outbound.parquet"

# All decision JSON exports — add new files to data/decisions/ and re-run
JSON_FILES = sorted(DECISIONS_DIR.glob("*.json"))
print(f"Found {len(JSON_FILES)} source file(s):")
for f in JSON_FILES:
    print(f"  {f.name}")

## Load & parse raw records

In [ ]:
all_rows = []

for filepath in JSON_FILES:
    print(f"
Processing {filepath.name}...")
    with open(filepath, encoding="utf-8") as f:
        total = sum(1 for _ in f)

    records = []
    with open(filepath, encoding="utf-8") as f:
        for line in tqdm(f, total=total, desc=filepath.name):
            records.append(json.loads(line))

    df_raw = pl.DataFrame(records)
    print(f"  {df_raw.shape[0]:,} raw interactions, {df_raw.shape[1]} columns")

    rows = extract_l5b15_rows(df_raw)   # extracts all LUGGAGE_VARIANTS by default
    print(f"  {len(rows):,} luggage scoring events extracted")
    all_rows.extend(rows)

print(f"
Total rows before deduplication: {len(all_rows):,}")

## Deduplicate & filter

In [ ]:
df_all = pd.DataFrame(all_rows)

# Deduplicate on modelExecutionID (safe to re-run with overlapping exports)
before = len(df_all)
df_all = df_all.drop_duplicates(subset=["modelExecutionID"])
print(f"After dedup: {len(df_all):,} rows  (removed {before - len(df_all):,} duplicates)")

# Filter to the channel/partition of interest
df = df_all[
    (df_all["pyChannel"]   == "Email") &
    (df_all["pyDirection"] == "Outbound") &
    (df_all["pyGroup"]     == "Luggage") &
    (df_all["pyIssue"]     == "Sales")
].copy().reset_index(drop=True)

# Parse interaction timestamp
df["pxDecisionTime"] = pd.to_datetime(
    df["pxDecisionTime"], format="%Y%m%dT%H%M%S.%f %Z", utc=True, errors="coerce"
)

print(f"After filter (Email / Outbound / Luggage / Sales): {len(df):,} rows")
print(f"Columns: {df.shape[1]}")
df.head(3)

## Partition & model version overview

In [ ]:
# Product variant breakdown
print("Decisions per luggage variant:")
print(df["pyName"].value_counts(dropna=False).to_string())

print()
dt = df["pxDecisionTime"].dropna()
if len(dt):
    print(f"Interaction date range: {dt.min().date()} → {dt.max().date()}")

In [ ]:
# Propensity by variant
print("Propensity summary by pyName:")
print(df.groupby("pyName")["propensity"].describe().round(4).to_string())

## Save to Parquet

In [ ]:
df.to_parquet(OUTPUT_FILE, index=False)
print(f"Saved {len(df):,} rows → {OUTPUT_FILE}")
print(f"File size: {OUTPUT_FILE.stat().st_size / 1024:.1f} KB")